### Machine Translation


Login hugging face

In [4]:
from huggingface_hub import login
login()

### Configs

In [5]:
from transformers import ( BitsAndBytesConfig, M2M100Tokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq)
import torch
import pandas as pd
from peft import LoraConfig, get_peft_model

LANGONE_CODE = "en"
LANGTWO_CODE = "tl"
CORPORA_PATH = "dataset/en-to-tl-corpora.csv"
MODEL_NAME = "facebook/m2m100_1.2B"
# configs
# quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    # use nf4 instead of fp4 for quantization performance
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# lora config
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

### Initializing Model

initialize model for quantization and LORA for lower VRAM consumption


In [ ]:
# model config
model_name = "facebook/m2m100_1.2B"

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    quantization_config=bnb_config
)

model = get_peft_model(model,lora_config)
model.print_trainable_parameters()


trainable params: 2,359,296 || all params: 1,241,829,376 || trainable%: 0.1900


### Initialized Tokenizer and Import Dataset
Lets finetune the model to support cebuano to english

In [7]:
import datasets
tokenizer = M2M100Tokenizer.from_pretrained(model_name,src_lang=LANGONE_CODE)
tokenizer.tgt_lang = LANGTWO_CODE

df = pd.read_csv(CORPORA_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(df)

       Unnamed: 0                                     language1_text  \
0          965446  In the beginning God created the heaven and th...   
1          965447  And the earth was without form, and void; and ...   
2          965448  And God said, Let there be light: and there wa...   
3          965449  And God saw the light, that it was good: and G...   
4          965450  And God called the light Day, and the darkness...   
...           ...                                                ...   
29179      994625  And the Spirit and the bride say, Come. And le...   
29180      994626  For I testify unto every man that heareth the ...   
29181      994627  and if any man shall take away from the words ...   
29182      994628  He which testifieth these things saith, Surely...   
29183      994629  The grace of our Lord Jesus Christ be with you...   

                                          language2_text  
0      Nang pasimula, nilikha ng Diyos ang langit at ...  
1      Ang lupa a

In [8]:

def preprocess(batch):
    
    model_inputs = tokenizer(
        batch["language1_text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    labels = tokenizer(
        batch["language2_text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_df = df.sample(frac=0.8, random_state=42)
eval_df = df[~df.index.isin(train_df.index)]
train_df = train_df.reset_index(drop=True)

training_dataset = datasets.Dataset.from_pandas(train_df)
evaluation_dataset = datasets.Dataset.from_pandas(eval_df)
train_dataset_processed = training_dataset.map(preprocess, batched=True, remove_columns=['language1_text', 'language2_text','Unnamed: 0'])
eval_dataset_processed = evaluation_dataset.map(preprocess, batched=True, remove_columns=['language1_text', 'language2_text','Unnamed: 0'])



Map:   0%|          | 0/23347 [00:00<?, ? examples/s]

Map:   0%|          | 0/5837 [00:00<?, ? examples/s]

### Training


In [9]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=5e-5,
    logging_steps=1000,
    save_steps=2000,
    save_total_limit=2,
    predict_with_generate=True,
    bf16=True,
    report_to="none"
)

In [10]:
from transformers import Seq2SeqTrainer
import numpy as np
import evaluate

sacrebleu = evaluate.load("sacrebleu")


def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]

    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}

    return result

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest",  # pads to the longest sequence in the batch
    return_tensors="pt" # converts lists -> PyTorch tensors
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_processed,
    eval_dataset=eval_dataset_processed,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)


/tmp/ipykernel_1417642/1568665726.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [11]:
trainer.evaluate()

{'eval_loss': 10.872014999389648,
 'eval_model_preparation_time': 0.0038,
 'eval_bleu': 0.1768,
 'eval_gen_len': 44.0033,
 'eval_runtime': 3651.6268,
 'eval_samples_per_second': 1.598,
 'eval_steps_per_second': 0.4}